# Limpieza de datos — Vianka

Esta limpieza es **preliminar**: conserva todos los registros y marca los casos que requieren revisión. Las reglas viven en `src/limpieza.py`, compartiendo la misma estructura utilizada en el turno de Ricardo.

## 1. Preparación reproducible

El notebook solo ejecuta y presenta resultados. No contiene transformaciones duplicadas. Cada campo modificado conserva una columna `_ORIGINAL`; los problemas dudosos se representan mediante banderas y nunca se eliminan automáticamente.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.diagnostico import cargar_datos
from src.limpieza import limpiar_datos_preliminar

df = cargar_datos()
limpio_pre = limpiar_datos_preliminar(df)

print(f"Registros: {len(limpio_pre):,}")
print(f"Columnas: {len(df.columns)} originales -> {len(limpio_pre.columns)} tras la limpieza preliminar")

Registros: 11,890
Columnas: 19 originales -> 48 tras la limpieza preliminar


## 2. `CODIGO` y `DISTRITO`

`CODIGO` permanece como texto y únicamente se valida. En `DISTRITO`, las cadenas vacías quedan como `NA`; los dos formatos completos se conservan y los códigos que terminan en guion se marcan para revisión manual.

In [2]:
resumen_codigos = pd.DataFrame([
    {"indicador": "CODIGO con formato válido", "cantidad": int(limpio_pre["CODIGO_FORMATO_VALIDO"].sum())},
    {"indicador": "CODIGO duplicado", "cantidad": int(limpio_pre["CODIGO_DUPLICADO"].sum())},
    {"indicador": "DISTRITO faltante (incluye filas vacías)", "cantidad": int(limpio_pre["DISTRITO"].isna().sum())},
    {"indicador": "DISTRITO incompleto conservado", "cantidad": int(limpio_pre["DISTRITO_INCOMPLETO"].sum())},
])
display(resumen_codigos)
display(limpio_pre["DISTRITO_FORMATO"].value_counts(dropna=False).rename_axis("formato").to_frame("cantidad"))
display(limpio_pre.loc[limpio_pre["DISTRITO_REQUIERE_REVISION"], ["CODIGO", "DISTRITO_ORIGINAL", "DISTRITO", "archivo_origen", "fila_origen"]].head())

,indicador,cantidad
0,CODIGO con formato válido,11867
1,CODIGO duplicado,0
2,DISTRITO faltante (incluye filas vacías),555
3,DISTRITO incompleto conservado,70


,cantidad
formato,
extendido NN-NN-NNNN,6226
corto NN-NNN,5039
faltante,555
incompleto NN-,70


,CODIGO,DISTRITO_ORIGINAL,DISTRITO,archivo_origen,fila_origen
1352,00-01-0198-46,01-,01-,CIUDADCAPITAL.csv,30
1368,00-01-0215-46,01-,01-,CIUDADCAPITAL.csv,46
1404,00-01-0258-46,01-,01-,CIUDADCAPITAL.csv,82
1413,00-01-0267-46,01-,01-,CIUDADCAPITAL.csv,91
1416,00-01-0271-46,01-,01-,CIUDADCAPITAL.csv,94


## 3. `DEPARTAMENTO`, `MUNICIPIO` y `DEPARTAMENTAL`

Los departamentos y departamentales se normalizan con catálogos separados. `CIUDAD CAPITAL` pasa a departamento y municipio `Guatemala`, pero su zona original se conserva en `ZONA_CAPITAL`. Los municipios se normalizan sin coincidencia aproximada.

In [3]:
resumen_geografia = pd.DataFrame([
    {"indicador": "Departamentos canónicos distintos", "cantidad": limpio_pre["DEPARTAMENTO"].nunique()},
    {"indicador": "Filas con ZONA_CAPITAL conservada", "cantidad": int(limpio_pre["ZONA_CAPITAL"].notna().sum())},
    {"indicador": "Municipio Pachalum corregido", "cantidad": int(limpio_pre["MUNICIPIO"].eq("Pachalum").sum())},
    {"indicador": "Departamentales canónicas distintas", "cantidad": limpio_pre["DEPARTAMENTAL"].nunique()},
    {"indicador": "Valores fuera de catálogos cerrados", "cantidad": int(limpio_pre["DEPARTAMENTO_FUERA_CATALOGO"].sum() + limpio_pre["DEPARTAMENTAL_FUERA_CATALOGO"].sum())},
])
display(resumen_geografia)
display(limpio_pre.loc[limpio_pre["ZONA_CAPITAL"].notna(), ["DEPARTAMENTO_ORIGINAL", "DEPARTAMENTO", "MUNICIPIO_ORIGINAL", "MUNICIPIO", "ZONA_CAPITAL"]].drop_duplicates().head())

,indicador,cantidad
0,Departamentos canónicos distintos,22
1,Filas con ZONA_CAPITAL conservada,2161
2,Municipio Pachalum corregido,18
3,Departamentales canónicas distintas,26
4,Valores fuera de catálogos cerrados,0


,DEPARTAMENTO_ORIGINAL,DEPARTAMENTO,MUNICIPIO_ORIGINAL,MUNICIPIO,ZONA_CAPITAL
1324,CIUDAD CAPITAL,Guatemala,ZONA 1,Guatemala,Zona 1
2192,CIUDAD CAPITAL,Guatemala,ZONA 2,Guatemala,Zona 2
2310,CIUDAD CAPITAL,Guatemala,ZONA 3,Guatemala,Zona 3
2371,CIUDAD CAPITAL,Guatemala,ZONA 4,Guatemala,Zona 4
2377,CIUDAD CAPITAL,Guatemala,ZONA 5,Guatemala,Zona 5


## 4. Consistencia y duplicados

El prefijo de `CODIGO` se contrasta con el departamento y el municipio. Los duplicados exactos de las cinco variables se marcan; no se elimina ninguna fila. La similitud de nombres de establecimientos continúa en el turno de Ricardo.

In [4]:
resumen_consistencia = pd.DataFrame([
    {"indicador": "CODIGO contradice DEPARTAMENTO", "cantidad": int(limpio_pre["CODIGO_DEPARTAMENTO_CONSISTENTE"].eq(False).sum())},
    {"indicador": "Prefijo asociado a varios municipios", "cantidad": int(limpio_pre["PREFIJO_CODIGO_AMBIGUO"].sum())},
    {"indicador": "Duplicados exactos en variables de Vianka", "cantidad": int(limpio_pre["DUPLICADO_EXACTO_VIANKA"].sum())},
])
display(resumen_consistencia)

,indicador,cantidad
0,CODIGO contradice DEPARTAMENTO,0
1,Prefijo asociado a varios municipios,0
2,Duplicados exactos en variables de Vianka,0


## 5. Resumen consolidado del turno 1

Las cantidades se recalculan en cada ejecución para que la documentación permanezca reproducible.

In [5]:
resumen_turno1 = pd.DataFrame([
    {"variable": "CODIGO", "regla": "Validar formato y unicidad, sin transformar", "cantidad_afectada": int(limpio_pre["CODIGO_FORMATO_VALIDO"].sum())},
    {"variable": "DISTRITO", "regla": "Vacío -> NA", "cantidad_afectada": int(limpio_pre["DISTRITO"].isna().sum())},
    {"variable": "DISTRITO", "regla": "Incompleto -> revisión, sin imputar", "cantidad_afectada": int(limpio_pre["DISTRITO_INCOMPLETO"].sum())},
    {"variable": "DEPARTAMENTO", "regla": "Catálogo oficial y Ciudad Capital -> Guatemala", "cantidad_afectada": int(limpio_pre["DEPARTAMENTO_ORIGINAL"].fillna("").ne(limpio_pre["DEPARTAMENTO"].fillna("")).sum())},
    {"variable": "MUNICIPIO", "regla": "Normalización ortográfica y nivel geográfico", "cantidad_afectada": int(limpio_pre["MUNICIPIO_ORIGINAL"].fillna("").ne(limpio_pre["MUNICIPIO"].fillna("")).sum())},
    {"variable": "ZONA_CAPITAL", "regla": "Conservar zona como variable derivada", "cantidad_afectada": int(limpio_pre["ZONA_CAPITAL"].notna().sum())},
    {"variable": "DEPARTAMENTAL", "regla": "Catálogo administrativo propio", "cantidad_afectada": int(limpio_pre["DEPARTAMENTAL_ORIGINAL"].fillna("").ne(limpio_pre["DEPARTAMENTAL"].fillna("")).sum())},
])
display(resumen_turno1)

,variable,regla,cantidad_afectada
0,CODIGO,"Validar formato y unicidad, sin transformar",11867
1,DISTRITO,Vacío -> NA,555
2,DISTRITO,"Incompleto -> revisión, sin imputar",70
3,DEPARTAMENTO,Catálogo oficial y Ciudad Capital -> Guatemala,11867
4,MUNICIPIO,Normalización ortográfica y nivel geográfico,11867
5,ZONA_CAPITAL,Conservar zona como variable derivada,2161
6,DEPARTAMENTAL,Catálogo administrativo propio,11867
